# Model Evaluation for Deep Learning

This notebook provides a comprehensive guide to evaluating deep learning models, covering essential metrics, visualization techniques, and best practices for assessing model performance.

## 1. Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Deep learning frameworks
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import tensorflow.keras.backend as K

# Scikit-learn utilities for evaluation
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, 
    roc_curve, auc, precision_recall_curve, average_precision_score,
    roc_auc_score
)
from sklearn.calibration import calibration_curve

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec

# Set the seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# For better visualization
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

# Display settings
pd.set_option('display.max_columns', None)

## 2. Load and Prepare Example Dataset

For this notebook, we'll use the Fashion MNIST dataset, a common benchmark in deep learning. This dataset contains 70,000 grayscale images of 10 different clothing items, with 60,000 training examples and 10,000 test examples.

In [ ]:
# Load Fashion MNIST dataset
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Print basic dataset information
print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

# Class names for Fashion MNIST
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Display sample distribution
plt.figure(figsize=(10, 6))
sns.countplot(x=y_train, palette='viridis')
plt.title('Distribution of Classes in Training Set')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(range(10), class_names, rotation=45)
plt.tight_layout()
plt.show()

### Data Preprocessing

Before feeding the data into our deep learning model, we need to:
1. Normalize the pixel values to be between 0 and 1
2. Reshape the data to include the channel dimension
3. Create a validation set from the training data

In [ ]:
# Normalize pixel values to be between 0 and 1
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Reshape data to include channel dimension (required for Conv2D layers)
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1)

# Split training data to create a validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Display sample images
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_train[i].reshape(28, 28), cmap='gray')
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Training set: {X_train.shape} samples")
print(f"Validation set: {X_val.shape} samples")
print(f"Test set: {X_test.shape} samples")

## 3. Building a Simple Deep Learning Model

Let's build a simple convolutional neural network (CNN) for image classification:

In [ ]:
def create_model():
    """Create a simple CNN model for Fashion MNIST classification"""
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    
    # Compile model with Adam optimizer
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Create the model
model = create_model()

# Display model architecture
model.summary()

In [ ]:
# Define callbacks for training
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.0001
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## 4. Common Evaluation Metrics

After training a deep learning model, the first step in evaluation is to examine basic performance metrics. For classification tasks, these typically include:

- **Accuracy**: The proportion of correct predictions among the total predictions
- **Precision**: The proportion of true positives among all positive predictions
- **Recall**: The proportion of true positives among all actual positives
- **F1-Score**: The harmonic mean of precision and recall

In [ ]:
# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Get predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate metrics
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Display metrics
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Value': [test_accuracy, precision, recall, f1]
})

print("\nEvaluation Metrics:")
print(metrics_df)

# Create a bar chart of metrics
plt.figure(figsize=(10, 6))
sns.barplot(x='Metric', y='Value', data=metrics_df, palette='viridis')
plt.title('Evaluation Metrics', fontsize=16)
plt.ylabel('Score', fontsize=14)
plt.ylim(0, 1)
for i, v in enumerate(metrics_df['Value']):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center')
plt.tight_layout()
plt.show()

## 5. Loss and Accuracy Curves

Plotting the training and validation loss/accuracy over epochs helps us diagnose overfitting, underfitting, and other training issues.

In [ ]:
def plot_learning_curves(history):
    """Plot training & validation accuracy and loss values"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot accuracy
    ax1.plot(history.history['accuracy'], label='Training Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Model Accuracy', fontsize=16)
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend(loc='lower right')
    ax1.grid(True)
    
    # Plot loss
    ax2.plot(history.history['loss'], label='Training Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_title('Model Loss', fontsize=16)
    ax2.set_ylabel('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend(loc='upper right')
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

# Plot learning curves
plot_learning_curves(history)

### Interpreting Learning Curves

When analyzing learning curves:

1. **Overfitting**: If training accuracy is high but validation accuracy is low, or if the validation loss starts to increase while training loss continues to decrease.

2. **Underfitting**: If both training and validation accuracy are low, suggesting the model isn't complex enough for the task.

3. **Good fit**: When training and validation metrics are close and have plateaued at acceptable levels.

4. **Early stopping point**: The epoch where validation loss is at its minimum, before overfitting begins.

## 6. Confusion Matrix and Classification Report

A confusion matrix helps visualize the performance of a classification algorithm. Each row represents the instances of an actual class, and each column represents the instances of a predicted class.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=18)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Normalize confusion matrix by row (true labels)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot normalized confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix', fontsize=18)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Generate classification report
report = classification_report(y_test, y_pred, target_names=class_names)
print("Classification Report:\n", report)

## 7. ROC Curves and AUC

ROC (Receiver Operating Characteristic) curves and AUC (Area Under the Curve) are useful metrics for evaluating binary classifiers. For multi-class classification, we can use a one-vs-rest approach to calculate ROC curves for each class.

In [ ]:
# Compute ROC curve and ROC area for each class
n_classes = 10
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(
        (y_test == i).astype(int), 
        y_pred_probs[:, i]
    )
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
plt.figure(figsize=(12, 10))

colors = plt.cm.viridis(np.linspace(0, 1, n_classes))

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'ROC curve of class {class_names[i]} (area = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.title('Receiver Operating Characteristic (ROC) Curve for Each Class', fontsize=16)
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

We can also compute the micro-averaged and macro-averaged ROC curves to get an overall sense of classifier performance across all classes.

In [ ]:
# Compute micro-average ROC curve and ROC area
y_test_binary = keras.utils.to_categorical(y_test, n_classes)
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_binary.ravel(), y_pred_probs.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Compute macro-average ROC curve and ROC area
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# Plot ROC curves
plt.figure(figsize=(10, 8))

plt.plot(fpr["micro"], tpr["micro"],
         label=f'micro-average ROC curve (area = {roc_auc["micro"]:.2f})',
         color='deeppink', linestyle=':', linewidth=4)

plt.plot(fpr["macro"], tpr["macro"],
         label=f'macro-average ROC curve (area = {roc_auc["macro"]:.2f})',
         color='navy', linestyle=':', linewidth=4)

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.title('Average ROC Curves', fontsize=16)
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Cross-Validation for Deep Learning

K-fold cross-validation is a technique to assess how well a model generalizes to an independent dataset. While it's computationally expensive for deep learning, it can be particularly useful when dealing with limited data.

In [ ]:
# For demonstration purposes, we'll use a smaller subset of the data
# and a simpler model for cross-validation
X_sample = X_train[:5000]
y_sample = y_train[:5000]

def create_simple_model():
    """Create a simpler model for cross-validation demonstration"""
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Define the K-fold Cross Validator
k_folds = 5
kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Initialize lists to store scores
cv_scores = []
fold_histories = []

# K-fold Cross Validation model evaluation
fold_no = 1
for train_idx, val_idx in kfold.split(X_sample):
    print(f'Training fold {fold_no}...')
    
    # Split data
    X_train_fold = X_sample[train_idx]
    y_train_fold = y_sample[train_idx]
    X_val_fold = X_sample[val_idx]
    y_val_fold = y_sample[val_idx]
    
    # Create and train model
    fold_model = create_simple_model()
    fold_history = fold_model.fit(
        X_train_fold, y_train_fold,
        epochs=10,
        batch_size=64,
        validation_data=(X_val_fold, y_val_fold),
        verbose=0
    )
    
    # Evaluate on validation data
    scores = fold_model.evaluate(X_val_fold, y_val_fold, verbose=0)
    print(f'Score for fold {fold_no}: {fold_model.metrics_names[0]} of {scores[0]}; {fold_model.metrics_names[1]} of {scores[1]*100:.2f}%')
    
    cv_scores.append(scores[1])
    fold_histories.append(fold_history)
    fold_no += 1

# Print average accuracy
print(f'Average accuracy across all folds: {np.mean(cv_scores)*100:.2f}% (±{np.std(cv_scores)*100:.2f}%)')

In [ ]:
# Visualize cross-validation results
plt.figure(figsize=(12, 5))

# Plot the validation accuracy for each fold
plt.subplot(1, 2, 1)
for i, history in enumerate(fold_histories):
    plt.plot(history.history['val_accuracy'], label=f'Fold {i+1}')

plt.title('Cross-Validation: Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)

# Plot the validation loss for each fold
plt.subplot(1, 2, 2)
for i, history in enumerate(fold_histories):
    plt.plot(history.history['val_loss'], label=f'Fold {i+1}')

plt.title('Cross-Validation: Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 9. Precision-Recall Curves

Precision-recall curves are particularly useful when dealing with imbalanced datasets, where accuracy might be misleading. They show the trade-off between precision and recall at different threshold settings.

In [ ]:
# Compute precision-recall curve for each class
precision = dict()
recall = dict()
avg_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(
        (y_test == i).astype(int), 
        y_pred_probs[:, i]
    )
    avg_precision[i] = average_precision_score(
        (y_test == i).astype(int), 
        y_pred_probs[:, i]
    )

# Plot precision-recall curve for selected classes
plt.figure(figsize=(12, 8))

# Choose a few representative classes to display
selected_classes = [0, 1, 4, 7, 9]
colors = plt.cm.viridis(np.linspace(0, 1, len(selected_classes)))

for i, color in zip(selected_classes, colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label=f'{class_names[i]} (AP = {avg_precision[i]:.2f})')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize=14)
plt.ylabel('Precision', fontsize=14)
plt.title('Precision-Recall Curves for Selected Classes', fontsize=16)
plt.legend(loc="lower left")
plt.grid(True)
plt.tight_layout()
plt.show()

## 10. Handling Class Imbalance in Evaluation

When dealing with imbalanced datasets, standard accuracy can be misleading. Let's create an imbalanced version of our dataset to demonstrate how to properly evaluate models in such scenarios.

In [ ]:
# Let's create an imbalanced subset of our data
# We'll keep only a small percentage of classes 0 and 1
def create_imbalanced_dataset():
    # Get indices for each class
    indices_by_class = {}
    for i in range(10):
        indices_by_class[i] = np.where(y_train == i)[0]
    
    # Keep only 10% of classes 0 and 1
    selected_indices = []
    for i in range(10):
        if i in [0, 1]:
            # Keep only 10% of these classes
            selected = indices_by_class[i][:int(len(indices_by_class[i]) * 0.1)]
        else:
            # Keep all samples from other classes
            selected = indices_by_class[i]
        selected_indices.extend(selected)
    
    # Create imbalanced dataset
    X_imbalanced = X_train[selected_indices]
    y_imbalanced = y_train[selected_indices]
    
    return X_imbalanced, y_imbalanced

X_imbalanced, y_imbalanced = create_imbalanced_dataset()

# Display the class distribution
plt.figure(figsize=(10, 6))
sns.countplot(x=y_imbalanced, palette='viridis')
plt.title('Distribution of Classes in Imbalanced Dataset')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(range(10), class_names, rotation=45)
plt.tight_layout()
plt.show()

# Split into training and validation sets
X_imb_train, X_imb_val, y_imb_train, y_imb_val = train_test_split(
    X_imbalanced, y_imbalanced, test_size=0.2, random_state=42, stratify=y_imbalanced
)

In [ ]:
# Create a model with class weights
def create_weighted_model():
    """Create a model that accounts for class imbalance"""
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Calculate class weights
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_imbalanced),
    y=y_imbalanced
)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

print("Class weights:")
for i, weight in class_weight_dict.items():
    print(f"Class {i} ({class_names[i]}): {weight:.4f}")

# Train the model with class weights
imbalanced_model = create_weighted_model()
history_imbalanced = imbalanced_model.fit(
    X_imb_train, y_imb_train,
    epochs=15,
    batch_size=128,
    validation_data=(X_imb_val, y_imb_val),
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# Evaluate the model on the test set
y_imb_pred = np.argmax(imbalanced_model.predict(X_test), axis=1)

# Calculate and display various metrics
print("Evaluation metrics for imbalanced data model:")
print("\nAccuracy:", accuracy_score(y_test, y_imb_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_imb_pred, target_names=class_names))

# Calculate metrics per class
precision_per_class = precision_score(y_test, y_imb_pred, average=None)
recall_per_class = recall_score(y_test, y_imb_pred, average=None)
f1_per_class = f1_score(y_test, y_imb_pred, average=None)

# Create a DataFrame for class metrics
class_metrics = pd.DataFrame({
    'Class': class_names,
    'Precision': precision_per_class,
    'Recall': recall_per_class,
    'F1-Score': f1_per_class
})

print("\nMetrics per class:")
print(class_metrics)

# Plot the class metrics
plt.figure(figsize=(12, 10))
class_metrics_melted = pd.melt(class_metrics, id_vars=['Class'], 
                               value_vars=['Precision', 'Recall', 'F1-Score'])
sns.barplot(x='Class', y='value', hue='variable', data=class_metrics_melted)
plt.title('Precision, Recall, and F1-Score by Class')
plt.xlabel('Class')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 11. Model Calibration

Model calibration refers to how well a model's predicted probabilities align with the actual outcome probabilities. A well-calibrated model should give predictions that match the true likelihood of events.

In [ ]:
# For binary classification calibration, we'll focus on a single class
# We'll treat class 0 (T-shirt/top) vs all other classes
y_test_binary = (y_test == 0).astype(int)
y_pred_probs_binary = y_pred_probs[:, 0]

# Generate reliability diagram
prob_true, prob_pred = calibration_curve(y_test_binary, y_pred_probs_binary, n_bins=10)

# Plot calibration curve
plt.figure(figsize=(10, 8))
plt.plot(prob_pred, prob_true, 's-', label='Calibration curve')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfectly calibrated')

# Calculate calibration metrics
from sklearn.metrics import brier_score_loss
brier = brier_score_loss(y_test_binary, y_pred_probs_binary)

plt.title(f'Calibration Plot (Brier Score: {brier:.4f})')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

### Calibration for All Classes

We can extend calibration analysis to all classes in our multi-class problem:

In [ ]:
# Calculate calibration curve for multiple classes
plt.figure(figsize=(12, 10))

for i in range(5):  # Show calibration for first 5 classes
    y_test_bin = (y_test == i).astype(int)
    y_prob_bin = y_pred_probs[:, i]
    
    prob_true, prob_pred = calibration_curve(y_test_bin, y_prob_bin, n_bins=10)
    brier = brier_score_loss(y_test_bin, y_prob_bin)
    
    plt.plot(prob_pred, prob_true, 's-', label=f'{class_names[i]} (Brier: {brier:.4f})')

plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfectly calibrated')
plt.title('Calibration Plot for Multiple Classes')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

## 12. Evaluation for Different Deep Learning Tasks

Different deep learning tasks require specialized evaluation metrics. Here's an example of a segmentation metric: Intersection over Union (IoU).

In [ ]:
# Define IoU metric
def iou_metric(y_true, y_pred):
    """Calculate IoU (Intersection over Union)"""
    # Convert predictions to binary masks
    y_pred = K.cast(K.greater(y_pred, 0.5), K.floatx())
    
    # Calculate intersection and union
    intersection = K.sum(y_true * y_pred, axis=[1, 2, 3])
    union = K.sum(y_true, axis=[1, 2, 3]) + K.sum(y_pred, axis=[1, 2, 3]) - intersection
    
    # Return IoU
    return K.mean((intersection + K.epsilon()) / (union + K.epsilon()))

# Define Dice coefficient
def dice_coefficient(y_true, y_pred):
    """Calculate Dice coefficient"""
    # Convert predictions to binary masks
    y_pred = K.cast(K.greater(y_pred, 0.5), K.floatx())
    
    # Calculate Dice coefficient: 2*|X∩Y|/(|X|+|Y|)
    intersection = K.sum(y_true * y_pred, axis=[1, 2, 3])
    union = K.sum(y_true, axis=[1, 2, 3]) + K.sum(y_pred, axis=[1, 2, 3])
    
    # Return Dice coefficient
    return K.mean((2. * intersection + K.epsilon()) / (union + K.epsilon()))

## 13. Ensemble Methods for Improved Evaluation

Ensemble methods combine multiple models to improve predictive performance. Let's create a simple ensemble and evaluate its performance.

In [ ]:
# Create and train multiple models for ensemble
def train_ensemble(n_models=3):
    models = []
    for i in range(n_models):
        print(f"Training model {i+1}/{n_models}...")
        model = create_model()
        model.fit(
            X_train, y_train,
            epochs=5,  # Fewer epochs for demonstration
            batch_size=128,
            validation_data=(X_val, y_val),
            verbose=0
        )
        models.append(model)
    return models

# Train the ensemble
ensemble_models = train_ensemble(3)

# Make predictions with the ensemble
ensemble_predictions = []
for model in ensemble_models:
    ensemble_predictions.append(model.predict(X_test))
    
# Average the predictions
ensemble_pred_probs = np.mean(ensemble_predictions, axis=0)
ensemble_pred = np.argmax(ensemble_pred_probs, axis=1)

# Calculate and display metrics
print("Ensemble Model Evaluation:")
print("\nAccuracy:", accuracy_score(y_test, ensemble_pred))
print("\nClassification Report:")
print(classification_report(y_test, ensemble_pred, target_names=class_names))

# Compare individual models with the ensemble
individual_accuracies = []
for i, model in enumerate(ensemble_models):
    y_pred_individual = np.argmax(model.predict(X_test), axis=1)
    acc = accuracy_score(y_test, y_pred_individual)
    individual_accuracies.append(acc)
    print(f"Model {i+1} Accuracy: {acc:.4f}")

# Compare with ensemble accuracy
ensemble_accuracy = accuracy_score(y_test, ensemble_pred)
print(f"Ensemble Accuracy: {ensemble_accuracy:.4f}")

# Plot comparison
plt.figure(figsize=(10, 6))
model_labels = [f"Model {i+1}" for i in range(len(ensemble_models))] + ["Ensemble"]
accuracies = individual_accuracies + [ensemble_accuracy]
plt.bar(model_labels, accuracies, color=['blue', 'green', 'orange', 'red'])
plt.title('Accuracy Comparison: Individual Models vs Ensemble')
plt.ylabel('Accuracy')
plt.ylim(0.8, 1.0)  # Adjust as needed
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.4f}", ha='center')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## 14. Early Stopping Implementation

Early stopping is a regularization technique that stops training when the validation performance stops improving, preventing overfitting.

In [ ]:
# Define more detailed early stopping callback
early_stopping_detailed = callbacks.EarlyStopping(
    monitor='val_loss',         # Metric to monitor
    min_delta=0.001,            # Minimum change to qualify as improvement
    patience=5,                 # Number of epochs with no improvement after which to stop
    verbose=1,                  # Verbosity mode
    restore_best_weights=True   # Restore model weights from the epoch with the best value
)

# Model checkpoint to save the best model
checkpoint = callbacks.ModelCheckpoint(
    'best_model.h5',            # Path to save the model
    monitor='val_loss',         # Metric to monitor
    save_best_only=True,        # Only save the best model
    verbose=1                   # Verbosity mode
)

# Define the model
model_with_early_stopping = create_model()

# Train with early stopping
history_es = model_with_early_stopping.fit(
    X_train, y_train,
    epochs=50,                  # Set a large number of epochs
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping_detailed, checkpoint],
    verbose=1
)

### Visualizing Early Stopping

Let's visualize how early stopping affects the training process:

In [ ]:
# Plot learning curves with early stopping point
def plot_early_stopping(history):
    """Plot learning curves with early stopping point"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Get the epoch where training stopped
    stopped_epoch = len(history.history['loss'])
    best_epoch = np.argmin(history.history['val_loss'])
    
    # Plot accuracy
    ax1.plot(history.history['accuracy'], label='Training Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.axvline(x=best_epoch, color='r', linestyle='--', label='Best Epoch')
    ax1.set_title(f'Model Accuracy (Best Epoch: {best_epoch+1})', fontsize=16)
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend(loc='lower right')
    ax1.grid(True)
    
    # Plot loss
    ax2.plot(history.history['loss'], label='Training Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.axvline(x=best_epoch, color='r', linestyle='--', label='Best Epoch')
    ax2.set_title(f'Model Loss (Best Epoch: {best_epoch+1})', fontsize=16)
    ax2.set_ylabel('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend(loc='upper right')
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return best_epoch

best_epoch = plot_early_stopping(history_es)

## 15. Deploying and Monitoring Models

After evaluation and fine-tuning, models need to be deployed to production environments. Here, we'll cover how to save models in different formats and discuss monitoring strategies.

In [ ]:
# Save model in different formats
# 1. Keras H5 format
model.save('fashion_mnist_model.h5')
print("Model saved in Keras H5 format.")

# 2. TensorFlow SavedModel format
model.save('fashion_mnist_savedmodel')
print("Model saved in TensorFlow SavedModel format.")

# 3. Convert to TensorFlow Lite (for mobile/edge devices)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('fashion_mnist_model.tflite', 'wb') as f:
    f.write(tflite_model)
print("Model converted to TensorFlow Lite format.")

### Model Monitoring

In production, models need to be monitored for:

1. **Performance drift**: When real-world data differs from training data
2. **Data drift**: Changes in the input distribution over time
3. **Concept drift**: Changes in the relationship between inputs and outputs
4. **System performance**: Latency, throughput, resource usage

Let's set up a basic monitoring system that records predictions and actual outcomes:

In [ ]:
# Simple monitoring system simulation
class ModelMonitor:
    def __init__(self, model_name):
        self.model_name = model_name
        self.predictions = []
        self.true_labels = []
        self.timestamps = []
        self.confidences = []
        
    def log_prediction(self, prediction, true_label=None, confidence=None):
        """Log a prediction with metadata"""
        import time
        self.predictions.append(prediction)
        self.true_labels.append(true_label)
        self.timestamps.append(time.time())
        self.confidences.append(confidence)
        
    def calculate_metrics(self):
        """Calculate monitoring metrics"""
        if None in self.true_labels:
            print("Some true labels are missing. Skipping accuracy calculation.")
            return None
        
        from sklearn.metrics import accuracy_score
        return {
            'accuracy': accuracy_score(self.true_labels, self.predictions),
            'total_predictions': len(self.predictions),
            'avg_confidence': np.mean(self.confidences) if None not in self.confidences else None
        }
        
    def visualize_monitoring(self):
        """Visualize monitoring data"""
        if None in self.true_labels:
            print("Some true labels are missing. Cannot visualize accuracy over time.")
            return
        
        import pandas as pd
        import matplotlib.pyplot as plt
        import matplotlib.dates as mdates
        from datetime import datetime
        
        # Convert timestamps to datetime
        dates = [datetime.fromtimestamp(ts) for ts in self.timestamps]
        
        # Create a dataframe with rolling accuracy
        df = pd.DataFrame({
            'timestamp': dates,
            'prediction': self.predictions,
            'true_label': self.true_labels,
            'correct': [p == t for p, t in zip(self.predictions, self.true_labels)]
        })
        
        # Calculate rolling accuracy
        df['rolling_accuracy'] = df['correct'].rolling(window=100, min_periods=1).mean()
        
        # Plot rolling accuracy over time
        plt.figure(figsize=(12, 6))
        plt.plot(df['timestamp'], df['rolling_accuracy'], 'b-')
        plt.title(f'Rolling Accuracy Over Time ({self.model_name})')
        plt.xlabel('Time')
        plt.ylabel('Rolling Accuracy (window=100)')
        plt.ylim(0, 1.05)
        plt.grid(True)
        plt.tight_layout()
        plt.show()

# Create a model monitor
monitor = ModelMonitor('Fashion MNIST Model')

# Simulate model serving
batch_size = 100
num_batches = 10

print("Simulating model serving...")
for batch in range(num_batches):
    # Get a batch of test data
    start_idx = (batch * batch_size) % len(X_test)
    end_idx = start_idx + batch_size
    batch_x = X_test[start_idx:end_idx]
    batch_y = y_test[start_idx:end_idx]
    
    # Make predictions
    batch_probs = model.predict(batch_x, verbose=0)
    batch_preds = np.argmax(batch_probs, axis=1)
    
    # Log each prediction
    for i in range(len(batch_preds)):
        confidence = np.max(batch_probs[i])
        monitor.log_prediction(batch_preds[i], batch_y[i], confidence)
    
    # Simulate some delay between batches
    import time
    time.sleep(0.5)

# Calculate and display metrics
metrics = monitor.calculate_metrics()
print("\nModel Monitoring Metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

# Visualize monitoring data
monitor.visualize_monitoring()

## Summary: Best Practices for Deep Learning Model Evaluation

1. **Use multiple evaluation metrics**:
   - Don't rely solely on accuracy
   - Consider precision, recall, F1-score, and AUC
   - Choose metrics appropriate for your problem domain

2. **Visualize model performance**:
   - Learning curves (loss and accuracy)
   - Confusion matrices
   - ROC and PR curves
   - Error analysis visualizations

3. **Address class imbalance**:
   - Use weighted metrics
   - Apply class weights during training
   - Consider resampling techniques
   - Use precision-recall curves instead of ROC curves

4. **Implement proper validation techniques**:
   - Train/validation/test splits
   - Cross-validation (when feasible)
   - Time-based validation for time series data

5. **Evaluate model calibration**:
   - Reliability diagrams
   - Calibration curves
   - Brier score

6. **Consider ensemble methods**:
   - Average predictions from multiple models
   - Use model stacking or bagging

7. **Implement early stopping properly**:
   - Monitor validation metrics
   - Set appropriate patience
   - Restore best weights

8. **Plan for deployment and monitoring**:
   - Save models in appropriate formats
   - Set up monitoring for performance drift
   - Establish retraining procedures

9. **Task-specific evaluation**:
   - Use IoU or Dice coefficient for segmentation
   - BLEU or ROUGE for text generation
   - mAP for object detection

By following these best practices, you can ensure that your deep learning models are thoroughly evaluated and ready for real-world applications.